# CleanPlate — click once, get a matte

Rotoscoping, matting and a comp, running on Colab's free GPU. Everything here is
the same code as the repo: [github.com/shauryadata/cleanplate](https://github.com/shauryadata/cleanplate).

**One thing to do by hand, now:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Then **Runtime → Run all**. It takes about 6–8 minutes, most of it downloading models.

It will: fetch the models, cut a 4-second shot out of *Tears of Steel*, track a subject
from two clicks, produce a soft matte, and composite it onto Mars. Then you can upload
your own clip and do the same.

Footage: (CC) Blender Foundation | mango.blender.org, CC BY 3.0. MatAnyone is
research/non-commercial (S-Lab 1.0) and is downloaded, never bundled — see THIRD_PARTY.md.


## 1. Check you actually got a GPU


In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
if out.returncode == 0 and out.stdout.strip():
    print('GPU:', out.stdout.strip())
else:
    print('NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all again.')
    print('It will still work on CPU, but expect roughly 10x the time.')


## 2. Get the code and the models

Pinned: the repo at a tag, SAM 2 and MatAnyone at the commits every published number
was produced with. Gradio is skipped — the app is not used here.


In [ ]:
CLEANPLATE_REF = 'main'   # the launch tag is set at release time

%cd /content
![ -d cleanplate ] || git clone --quiet https://github.com/shauryadata/cleanplate.git
%cd /content/cleanplate
!git checkout --quiet $CLEANPLATE_REF && git log -1 --format='repo at %h %s'
!grep -v '^gradio' requirements.txt > /tmp/req.txt
!pip install --quiet -r /tmp/req.txt
print('deps installed')


In [ ]:
# SAM 2 + MatAnyone at their pinned commits, the SAM 2.1 checkpoint, and the footage.
# SAM2_BUILD_CUDA=0: the optional CUDA kernels are slow to compile on Colab and the
# model runs fine without them.
import os
os.environ['SAM2_BUILD_CUDA'] = '0'
!bash scripts/download.sh sam2      2>&1 | tail -2
!bash scripts/download.sh checkpoints 2>&1 | tail -2
!bash scripts/download.sh matanyone 2>&1 | tail -2
!bash scripts/download.sh footage   2>&1 | tail -2


## 3. Cut the demo shot and look at frame 0

The grid is there so you can read off click coordinates for your own clip later.


In [ ]:
import sys; sys.path.insert(0, '/content/cleanplate')
from cleanplate import ingest
from PIL import Image, ImageDraw
import numpy as np

if not ingest.frame_paths('walk'):
    ingest.extract_shot('walk', start=270.0, duration=4.0, width=960)

def with_grid(img, step=120):
    im = Image.fromarray(np.asarray(img)).convert('RGB'); d = ImageDraw.Draw(im)
    for x in range(0, im.width, step):
        d.line([(x, 0), (x, im.height)], fill=(255, 140, 0)); d.text((x + 3, 3), str(x), fill=(255, 255, 0))
    for y in range(0, im.height, 100):
        d.line([(0, y), (im.width, y)], fill=(255, 140, 0)); d.text((3, y + 3), str(y), fill=(255, 255, 0))
    return im

with_grid(ingest.load_frame('walk', 0))


## 4. Click once. Get a matte and a comp.

Two clicks on the actor: one on the head, one on the body. Those two numbers are the
whole interface.


In [ ]:
!python scripts/colab_demo.py --shot walk --click 470,150 --click 455,300 \
    --bg datasets/backgrounds/mars_clean.jpg --out-name colab


In [ ]:
from IPython.display import HTML, display
import base64, pathlib

def show(path, width=900):
    b64 = base64.b64encode(pathlib.Path(path).read_bytes()).decode()
    display(HTML(f'<video width={width} controls autoplay loop muted '
                 f'src="data:video/mp4;base64,{b64}"></video>'))

display(Image.open('outputs/colab/contact.jpg'))   # plate | matte | comp
show('outputs/colab/comp.mp4')
show('outputs/colab/matte.mp4')


## 5. Your own clip

Run the next cell, pick a video (a few seconds is plenty), then read the click
coordinates off the grid and run the last cell.


In [ ]:
from google.colab import files
up = files.upload()
MY_VIDEO = '/content/cleanplate/' + list(up)[0]
print('uploaded:', MY_VIDEO)

START_SECONDS = 0.0   # where to start in your clip
SECONDS = 4.0         # how much to process

from cleanplate import ingest
ingest.extract_shot('myclip', start=START_SECONDS, duration=SECONDS,
                    source=MY_VIDEO, width=960, force=True)
with_grid(ingest.load_frame('myclip', 0))


In [ ]:
CLICK_X, CLICK_Y = 480, 200      # <- read these off the grid above
SECOND_CLICK = None              # e.g. (480, 330) for a second point on the body

extra = f' --click {SECOND_CLICK[0]},{SECOND_CLICK[1]}' if SECOND_CLICK else ''
cmd = (f'python scripts/colab_demo.py --shot myclip --click {CLICK_X},{CLICK_Y}'
       + extra + ' --bg datasets/backgrounds/mars_clean.jpg --out-name mycolab')
print(cmd)
!{cmd}
display(Image.open('outputs/mycolab/contact.jpg'))
show('outputs/mycolab/comp.mp4')


---
**If a click grabbed the wrong thing**, pick a point closer to the middle of the subject
and add a second click on another part of them. Two clicks on one person beat one click
every time — a single point can land on a jacket and select only the jacket.

Full app (click-by-click corrections, removal, export): run it locally with `python app.py`.
